# CrisisWorld Router SFT Demo (Google Colab)

This notebook demonstrates supervised fine-tuning of the **Cortex Executive** router
using Unsloth + LoRA on Llama 3.1 8B (4-bit quantized).

**What it does:**
1. Installs CrisisWorld from git
2. Loads the bundled `router_sft.jsonl` training data
3. Runs a short SFT with Unsloth (~60 steps)
4. Compares pre- vs post-SFT offline metrics (JSON validity, route accuracy)
5. Runs CrisisWorld episodes comparing heuristic vs. tuned Executive

**Requirements:** GPU runtime (T4 is sufficient). Go to Runtime > Change runtime type > T4 GPU.

**Estimated time:** ~15 minutes on T4.

In [ ]:
# Cell 1: User Configuration
# All tunable knobs in one place.

REPO_URL = "https://github.com/YOUR_USERNAME/MetaFinals.git"  # <-- change this
BASE_MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024
TRAIN_ROWS = 256          # rows from router_sft.jsonl for training
VAL_ROWS = 64             # held-out rows for validation
MAX_STEPS = 60            # short demo run
LORA_R = 16
LORA_ALPHA = 32
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM = 4
EPISODE_SEEDS = [101, 102, 103]
EPISODE_MAX_TURNS = 10
OUTPUT_DIR = "outputs/router_sft_demo"

In [ ]:
# Cell 2: Install dependencies
# Install CrisisWorld from git (provides all project modules + bundled data).
# Then install Unsloth and training dependencies separately.

!pip install -q "git+{REPO_URL}" 2>&1 | tail -3
!pip install -q unsloth trl peft bitsandbytes datasets accelerate matplotlib seaborn 2>&1 | tail -3
print("Installation complete.")

In [ ]:
# Cell 3: Verify installation + GPU check
import torch
assert torch.cuda.is_available(), (
    "No GPU detected! Go to Runtime > Change runtime type > T4 GPU."
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

import CrisisWorld
pkg_root = CrisisWorld.get_package_root()
print(f"\nCrisisWorld installed at: {pkg_root}")

data_path = pkg_root / "data" / "router_sft.jsonl"
assert data_path.exists(), f"Data file not found at {data_path}"
print(f"Router SFT data: {data_path}")

# Verify key imports
from CrisisWorld.training.router_sft import ROUTE_LABELS, canonical_route_label
print(f"Route labels: {ROUTE_LABELS}")
print("\nAll checks passed.")

In [ ]:
# Cell 4: HuggingFace authentication
# Llama 3.1 is a gated model. You must:
# 1. Accept the license at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
# 2. Log in with your HF token below.

from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Cell 5: Load and prepare dataset
import json
import random
from collections import Counter
from datasets import Dataset

import CrisisWorld
from CrisisWorld.training.router_sft import canonical_route_label

data_path = CrisisWorld.get_package_root() / "data" / "router_sft.jsonl"
rows = [json.loads(line) for line in data_path.read_text().strip().split("\n") if line.strip()]
print(f"Total rows in router_sft.jsonl: {len(rows)}")

# Extract route labels from assistant responses
for row in rows:
    assistant_text = row["messages"][-1]["content"]
    try:
        decision = json.loads(assistant_text)
        row["route_label"] = canonical_route_label(decision)
    except (json.JSONDecodeError, ValueError, KeyError):
        row["route_label"] = "unknown"

# Deterministic shuffle and split
rng = random.Random(42)
rng.shuffle(rows)
train_rows = rows[:TRAIN_ROWS]
val_rows = rows[TRAIN_ROWS:TRAIN_ROWS + VAL_ROWS]
print(f"Train: {len(train_rows)}, Val: {len(val_rows)}")

# Show label distribution
train_labels = Counter(r["route_label"] for r in train_rows)
print(f"\nTrain label distribution:")
for label, count in sorted(train_labels.items()):
    print(f"  {label}: {count}")

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)

# Show a sample
print(f"\nSample assistant response:")
print(train_rows[0]["messages"][-1]["content"][:200])

In [ ]:
# Cell 6: Pre-SFT evaluation helpers
import json
import re

from CrisisWorld.training.router_sft import ROUTE_LABELS, canonical_route_label


def extract_json_from_text(text: str) -> dict | None:
    """Extract first JSON object from model output text."""
    stripped = re.sub(r"```(?:json)?\s*", "", text).replace("```", "")
    start = stripped.find("{")
    if start == -1:
        return None
    depth = 0
    in_string = False
    escape = False
    for i in range(start, len(stripped)):
        c = stripped[i]
        if escape:
            escape = False
            continue
        if c == "\\":
            escape = True
            continue
        if c == '"' and not escape:
            in_string = not in_string
            continue
        if in_string:
            continue
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(stripped[start:i+1])
                except json.JSONDecodeError:
                    return None
    return None


def get_route_label(text: str) -> str | None:
    """Extract and normalize route label from model output."""
    parsed = extract_json_from_text(text)
    if parsed is None:
        return None
    try:
        return canonical_route_label(parsed)
    except (ValueError, KeyError):
        return None


def evaluate_on_prompts(model, tokenizer, prompts: list[dict], labels: list[str]) -> dict:
    """Run offline evaluation: JSON validity rate and route-label accuracy."""
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)

    valid_json = 0
    correct_route = 0
    total = len(prompts)
    sample_outputs = []

    for i, (prompt, gold_label) in enumerate(zip(prompts, labels)):
        messages = prompt["messages"][:-1]  # system + user (drop assistant)
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_new_tokens=256, temperature=0.1,
                do_sample=True, pad_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )

        parsed = extract_json_from_text(generated)
        is_valid = parsed is not None
        pred_label = get_route_label(generated)
        is_correct = pred_label == gold_label

        if is_valid:
            valid_json += 1
        if is_correct:
            correct_route += 1

        if i < 3:  # save first 3 for display
            sample_outputs.append({
                "gold": gold_label,
                "pred": pred_label,
                "output": generated[:200],
            })

    return {
        "json_valid_rate": valid_json / total if total > 0 else 0.0,
        "route_accuracy": correct_route / total if total > 0 else 0.0,
        "total": total,
        "samples": sample_outputs,
    }


print("Evaluation helpers defined.")

In [ ]:
# Cell 7: Load Unsloth model + LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(f"Model loaded: {BASE_MODEL}")
print(f"LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")
trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Cell 8: Pre-SFT baseline evaluation
# Run evaluation on the base model before training to establish baseline.

import torch

eval_subset = val_rows[:16]  # small subset for speed
eval_labels = [r["route_label"] for r in eval_subset]

print("Running pre-SFT evaluation on base model...")
pre_sft_metrics = evaluate_on_prompts(model, tokenizer, eval_subset, eval_labels)

print(f"\nPre-SFT Results:")
print(f"  JSON valid rate: {pre_sft_metrics['json_valid_rate']:.1%}")
print(f"  Route accuracy:  {pre_sft_metrics['route_accuracy']:.1%}")
print(f"  Evaluated: {pre_sft_metrics['total']} prompts")

print(f"\nSample outputs:")
for s in pre_sft_metrics["samples"]:
    print(f"  gold={s['gold']:20s} pred={str(s['pred']):20s} | {s['output'][:80]}")

In [ ]:
# Cell 9: Prepare chat text for SFT
# Convert messages to a single text field using the tokenizer's chat template.

def format_chat(row):
    """Apply chat template to produce a single training string."""
    text = tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

train_ds_formatted = train_ds.map(format_chat)
val_ds_formatted = val_ds.map(format_chat)

print(f"Train: {len(train_ds_formatted)} examples")
print(f"Val:   {len(val_ds_formatted)} examples")
print(f"\nSample text (first 300 chars):")
print(train_ds_formatted[0]["text"][:300])

In [ ]:
# Cell 10: Configure and run SFT training
import time
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel

# Switch back to training mode
FastLanguageModel.for_training(model)

# T4 (compute capability 7.5) only supports fp16, not bf16.
# bf16 requires Ampere+ (compute capability >= 8.0).
use_bf16 = torch.cuda.get_device_capability()[0] >= 8
use_fp16 = not use_bf16
print(f"GPU precision: {'bf16' if use_bf16 else 'fp16'}")

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=5,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=2,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="no",
    max_seq_length=MAX_SEQ_LENGTH,
    bf16=use_bf16,
    fp16=use_fp16,
    dataset_text_field="text",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_formatted,
    eval_dataset=val_ds_formatted,
    tokenizer=tokenizer,
)

print(f"Starting SFT: {MAX_STEPS} steps, batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}")
start_time = time.time()
trainer.train()
elapsed = time.time() - start_time

print(f"\nTraining complete in {elapsed/60:.1f} minutes.")
final_log = [e for e in trainer.state.log_history if "train_loss" in e]
if final_log:
    print(f"Final train loss: {final_log[-1]['train_loss']:.4f}")
eval_logs = [e for e in trainer.state.log_history if "eval_loss" in e]
if eval_logs:
    print(f"Final eval loss:  {eval_logs[-1]['eval_loss']:.4f}")

In [ ]:
# Cell 11: Save adapter
import os

adapter_path = os.path.join(OUTPUT_DIR, "adapter_final")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

# Save training log
import json
log_path = os.path.join(OUTPUT_DIR, "training_log.json")
with open(log_path, "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)
print(f"Training log saved to {log_path}")

In [ ]:
# Cell 12: Plot training curves
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_steps = [e["step"] for e in history if "loss" in e]
train_loss = [e["loss"] for e in history if "loss" in e]
eval_steps = [e["step"] for e in history if "eval_loss" in e]
eval_loss = [e["eval_loss"] for e in history if "eval_loss" in e]
lr_steps = [e["step"] for e in history if "learning_rate" in e]
lr_vals = [e["learning_rate"] for e in history if "learning_rate" in e]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Train loss
axes[0].plot(train_steps, train_loss, "b-", alpha=0.7, label="Train")
if eval_loss:
    axes[0].plot(eval_steps, eval_loss, "ro-", markersize=5, label="Eval")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Learning rate
axes[1].plot(lr_steps, lr_vals, "g-")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Learning Rate")
axes[1].set_title("Learning Rate Schedule")
axes[1].grid(True, alpha=0.3)

# Loss delta
if len(train_loss) >= 2:
    first_loss = sum(train_loss[:3]) / min(3, len(train_loss))
    last_loss = sum(train_loss[-3:]) / min(3, len(train_loss))
    axes[2].bar(["First 3 steps", "Last 3 steps"], [first_loss, last_loss],
               color=["salmon", "lightgreen"])
    axes[2].set_ylabel("Avg Loss")
    axes[2].set_title(f"Loss Change: {first_loss:.3f} -> {last_loss:.3f}")
    axes[2].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.show()
print("Training curves saved.")

In [ ]:
# Cell 13: Post-SFT offline evaluation
# Compare base model (pre-SFT) vs. trained adapter (post-SFT).

import torch

print("Running post-SFT evaluation on tuned model...")
post_sft_metrics = evaluate_on_prompts(model, tokenizer, eval_subset, eval_labels)

print(f"\n{'Metric':<25} {'Pre-SFT':>10} {'Post-SFT':>10} {'Delta':>10}")
print("-" * 58)

pre_json = pre_sft_metrics["json_valid_rate"]
post_json = post_sft_metrics["json_valid_rate"]
print(f"{'JSON valid rate':<25} {pre_json:>9.1%} {post_json:>9.1%} {post_json - pre_json:>+9.1%}")

pre_acc = pre_sft_metrics["route_accuracy"]
post_acc = post_sft_metrics["route_accuracy"]
print(f"{'Route accuracy':<25} {pre_acc:>9.1%} {post_acc:>9.1%} {post_acc - pre_acc:>+9.1%}")

print(f"\nPost-SFT sample outputs:")
for s in post_sft_metrics["samples"]:
    print(f"  gold={s['gold']:20s} pred={str(s['pred']):20s} | {s['output'][:80]}")

In [ ]:
# Cell 14: Build notebook-local Executive for episode evaluation
# Replaces only the Executive role with the tuned model.
# All other roles (Perception, WorldModeler, Planner, Critic) stay heuristic.

import json
import logging
import torch
from unsloth import FastLanguageModel
from CrisisWorld.training.router_sft import canonical_route_label
from CrisisWorld.training.data_export import ROUTER_SYSTEM

_log = logging.getLogger("notebook_executive")


class NotebookExecutiveRole:
    """Executive role backed by the trained LoRA adapter.

    Falls back to heuristic ExecutiveRole on JSON parse failure.
    """

    def __init__(self, model, tokenizer, fallback_role):
        self._model = model
        self._tokenizer = tokenizer
        self._fallback = fallback_role
        self._cost = 1
        self.total_calls = 0
        self.fallback_count = 0
        FastLanguageModel.for_inference(model)

    @property
    def role_name(self) -> str:
        return "executive"

    @property
    def cost(self) -> int:
        return self._cost

    def invoke(self, role_input):
        """Generate executive decision via the tuned model."""
        self.total_calls += 1
        # Build prompt from role_input payload
        payload = role_input.payload
        user_text = ""
        if isinstance(payload, dict):
            user_text = json.dumps(payload, default=str)[:1500]
        elif isinstance(payload, str):
            user_text = payload[:1500]
        else:
            user_text = str(payload)[:1500]

        messages = [
            {"role": "system", "content": ROUTER_SYSTEM},
            {"role": "user", "content": user_text},
        ]
        input_text = self._tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self._tokenizer(input_text, return_tensors="pt").to(self._model.device)

        with torch.no_grad():
            output_ids = self._model.generate(
                **inputs, max_new_tokens=256, temperature=0.1,
                do_sample=True, pad_token_id=self._tokenizer.eos_token_id,
            )
        generated = self._tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )

        # Try to parse as ExecutiveDecision
        parsed = extract_json_from_text(generated)
        if parsed is not None and "decision" in parsed:
            try:
                from CrisisWorld.schemas.artifact import ExecutiveDecision
                return ExecutiveDecision.model_validate(parsed)
            except Exception:
                pass

        # Fallback to heuristic
        self.fallback_count += 1
        _log.debug("LLM executive parse failed, using heuristic fallback")
        return self._fallback.invoke(role_input)


print("NotebookExecutiveRole defined.")
print("This wraps the tuned model with a fallback to the heuristic Executive.")

In [ ]:
# Cell 15: Run CrisisWorld episodes
# Compare heuristic Executive vs. tuned NotebookExecutiveRole.

import numpy as np
from collections import defaultdict

from CrisisWorld.server import CrisisWorld
from CrisisWorld.cortex import BudgetTracker, CortexDeliberator, EpisodeMemory
from CrisisWorld.cortex.roles import (
    PerceptionRole, WorldModelerRole, PlannerRole, CriticRole, ExecutiveRole,
)
from CrisisWorld.agents import CortexAgent
from CrisisWorld.tracing import EpisodeTracer
from CrisisWorld.models import EnvConfig


def run_episode(env, agent, seed, max_turns):
    """Run a single episode, return cumulative reward and stats."""
    import uuid
    episode_id = str(uuid.uuid4())
    obs = env.reset(seed=seed, episode_id=episode_id)
    agent.reset()
    cumulative_reward = 0.0
    decisions = []

    for turn in range(max_turns):
        action = agent.act(obs)
        obs = env.step(action)
        cumulative_reward += obs.reward if obs.reward else 0.0
        decisions.append(getattr(action, "kind", "unknown"))
        if obs.done:
            break

    return {
        "seed": seed,
        "cumulative_reward": cumulative_reward,
        "turns": turn + 1,
        "done": obs.done,
        "decisions": decisions,
    }


# Build environment
env_config = EnvConfig()
env = CrisisWorld(config=env_config)

# --- Condition 1: Heuristic Executive ---
print("Running episodes with HEURISTIC Executive...")
heuristic_results = []
for seed in EPISODE_SEEDS:
    heuristic_exec = ExecutiveRole()
    roles = {
        "perception": PerceptionRole(),
        "world_modeler": WorldModelerRole(),
        "planner": PlannerRole(),
        "critic": CriticRole(),
        "executive": heuristic_exec,
    }
    memory = EpisodeMemory()
    tracer = EpisodeTracer(episode_id=f"heuristic-{seed}")
    budget = BudgetTracker(total_budget=30)
    deliberator = CortexDeliberator(roles=roles, memory=memory, logger=tracer)
    agent = CortexAgent(deliberator=deliberator, budget=budget, logger=tracer)
    result = run_episode(env, agent, seed, EPISODE_MAX_TURNS)
    result["condition"] = "heuristic"
    heuristic_results.append(result)
    print(f"  seed={seed}: reward={result['cumulative_reward']:.3f}, turns={result['turns']}")

# --- Condition 2: Tuned Executive ---
print(f"\nRunning episodes with TUNED Executive...")
tuned_results = []
for seed in EPISODE_SEEDS:
    heuristic_fallback = ExecutiveRole()
    tuned_exec = NotebookExecutiveRole(model, tokenizer, heuristic_fallback)
    roles = {
        "perception": PerceptionRole(),
        "world_modeler": WorldModelerRole(),
        "planner": PlannerRole(),
        "critic": CriticRole(),
        "executive": tuned_exec,
    }
    memory = EpisodeMemory()
    tracer = EpisodeTracer(episode_id=f"tuned-{seed}")
    budget = BudgetTracker(total_budget=30)
    deliberator = CortexDeliberator(roles=roles, memory=memory, logger=tracer)
    agent = CortexAgent(deliberator=deliberator, budget=budget, logger=tracer)
    result = run_episode(env, agent, seed, EPISODE_MAX_TURNS)
    result["condition"] = "tuned"
    tuned_results.append(result)
    fb = tuned_exec.fallback_count
    tc = tuned_exec.total_calls
    print(f"  seed={seed}: reward={result['cumulative_reward']:.3f}, turns={result['turns']}, fallback={fb}/{tc}")

env.close()
print("\nEpisode evaluation complete.")

In [ ]:
# Cell 16: Plot episode comparison results
import matplotlib.pyplot as plt
import numpy as np

h_rewards = [r["cumulative_reward"] for r in heuristic_results]
t_rewards = [r["cumulative_reward"] for r in tuned_results]
h_turns = [r["turns"] for r in heuristic_results]
t_turns = [r["turns"] for r in tuned_results]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Cumulative reward by seed
x = np.arange(len(EPISODE_SEEDS))
width = 0.35
axes[0].bar(x - width/2, h_rewards, width, label="Heuristic", color="steelblue")
axes[0].bar(x + width/2, t_rewards, width, label="Tuned", color="coral")
axes[0].set_xlabel("Seed")
axes[0].set_ylabel("Cumulative Reward")
axes[0].set_title("Reward by Seed")
axes[0].set_xticks(x)
axes[0].set_xticklabels([str(s) for s in EPISODE_SEEDS])
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis="y")

# Mean reward with error bars
means = [np.mean(h_rewards), np.mean(t_rewards)]
stds = [np.std(h_rewards), np.std(t_rewards)]
axes[1].bar(["Heuristic", "Tuned"], means, yerr=stds,
           color=["steelblue", "coral"], capsize=10)
axes[1].set_ylabel("Mean Cumulative Reward")
axes[1].set_title(f"Mean Reward (n={len(EPISODE_SEEDS)})")
axes[1].grid(True, alpha=0.3, axis="y")

# Episode duration
axes[2].bar(x - width/2, h_turns, width, label="Heuristic", color="steelblue")
axes[2].bar(x + width/2, t_turns, width, label="Tuned", color="coral")
axes[2].set_xlabel("Seed")
axes[2].set_ylabel("Turns")
axes[2].set_title("Episode Duration")
axes[2].set_xticks(x)
axes[2].set_xticklabels([str(s) for s in EPISODE_SEEDS])
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "episode_comparison.png"), dpi=150)
plt.show()

# Summary table
print(f"\n{'Condition':<12} {'Mean Reward':>12} {'Std':>8} {'Mean Turns':>12}")
print("-" * 48)
print(f"{'Heuristic':<12} {np.mean(h_rewards):>12.3f} {np.std(h_rewards):>8.3f} {np.mean(h_turns):>12.1f}")
print(f"{'Tuned':<12} {np.mean(t_rewards):>12.3f} {np.std(t_rewards):>8.3f} {np.mean(t_turns):>12.1f}")

## Summary and Caveats

**What we showed:**
- SFT on the Executive router task with a short Unsloth training run
- Offline metrics: JSON validity rate and route-label accuracy before vs. after SFT
- Environment evaluation: heuristic vs. tuned Executive in CrisisWorld episodes

**Caveats (read these!):**
- This is a **tiny showcase run** (~60 steps). No claim of convergence.
- Reward movement can be **noisy with few seeds** (3 episodes per condition).
- SFT optimizes **imitation loss**, not reward directly. Improvements in episode reward
  come indirectly from better routing decisions.
- The tuned Executive falls back to the heuristic on JSON parse failures.
  High fallback rates mean the model needs more training data or steps.
- For a proper evaluation, run with more seeds, more training steps, and the full
  ablation conditions described in the project README.